# Smriti — Face Embedding Bridge

Run this notebook on **Kaggle** or **Colab** with a GPU runtime. It starts a FastAPI server that receives batches of 112×112 face crops from your desktop Smriti and returns 512-d embeddings.

**Model must match the desktop.** Smriti's default embedder is **AdaFace** (`adaface_ir101_webface12m.onnx`). If you changed it to `glintr100.onnx` in Settings, set `MODEL_NAME` in cell 2 to the matching model - the bridge's `/health` response advertises which model is loaded and Smriti refuses to send work when they don't match (mismatched models produce embeddings in different metric spaces, which would corrupt clustering).


In [ ]:
!pip install -q "numpy<2" fastapi uvicorn nest-asyncio python-multipart onnxruntime-gpu==1.18.0 pyngrok

In [ ]:
# Pick ONE model. Default = AdaFace (matches Smriti's default).
# To use glintr100 instead, set MODEL_NAME='glintr100'.
MODEL_NAME = 'adaface_ir101_webface12m'
MODEL_URLS = {
    'adaface_ir101_webface12m': 'https://drive.usercontent.google.com/download?id=1dgMFOASKnaujQcCL4sSYkKOkBrmXUUU1&export=download&confirm=t',
    'glintr100': 'https://huggingface.co/MonsterMMORPG/tools/resolve/main/glintr100.onnx',
}
MODEL_URL = MODEL_URLS[MODEL_NAME]

!curl -L "$MODEL_URL" -o /content/model.onnx
!ls -lh /content/model.onnx
assert __import__('os').path.getsize('/content/model.onnx') > 100_000_000, 'model download failed or returned an HTML error page'

In [ ]:
# Verify GPU is available
import onnxruntime as ort
sess = ort.InferenceSession('/content/model.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
print('GPU:', sess.get_providers())

In [ ]:
from fastapi import FastAPI, UploadFile, File
import numpy as np
from PIL import Image
import io

app = FastAPI()
session = sess  # reuse the verified session

def preprocess(img_bytes: bytes) -> np.ndarray:
    """Decode JPEG, resize to 112x112, normalize to [-1, 1], return [1, 3, 112, 112]."""
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img = img.resize((112, 112), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)
    arr = (arr - 127.5) / 127.5  # normalize to [-1, 1]
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    return np.expand_dims(arr, axis=0)  # [1, 3, 112, 112]

def l2_normalize(vec: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

@app.post('/embed')
async def embed(files: list[UploadFile] = File(...)):
    """Receive N face crops, return N 512-d embeddings."""
    if not files:
        return {'embeddings': []}
    face_bytes = []
    for f in files:
        face_bytes.append(await f.read())
    batch = np.concatenate([preprocess(b) for b in face_bytes], axis=0).astype(np.float32)
    onnx_inputs = {session.get_inputs()[0].name: batch}
    output = session.run(None, onnx_inputs)[0]
    embeddings = [l2_normalize(output[i]).tolist() for i in range(output.shape[0])]
    return {'embeddings': embeddings}

@app.get('/health')
async def health():
    providers = session.get_providers()
    return {
        'status': 'ok',
        'provider': providers[0] if providers else 'unknown',
        'model': MODEL_NAME,
    }

print(f'FastAPI app ready — serving model {MODEL_NAME!r}')

In [ ]:
# === Tunnel: ngrok (default) ===
# Sign up for free at https://ngrok.com — copy your auth token from the dashboard.
from pyngrok import ngrok
ngrok.set_auth_token('YOUR_TOKEN_HERE')  # <-- paste your ngrok authtoken
public_url = ngrok.connect(8000, 'http')
print(f'BRIDGE URL -> {public_url}')
print('Paste this URL into Smriti Settings -> Cloud face acceleration')

In [ ]:
# === Tunnel: cloudflared (alternative, no account needed) ===
# Uncomment the block below to use cloudflared instead of ngrok.
# The URL prints to cloudflared's stdout — watch the cell output.

# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
# !chmod +x cloudflared
# import subprocess, threading
# def run_tunnel():
#     subprocess.run(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], check=False)
# threading.Thread(target=run_tunnel, daemon=True).start()
# print('cloudflared tunnel starting — watch for: https://xxx-xxx-xxx.trycloudflare.com')
# import time; time.sleep(5)

In [ ]:
# === Run the server (non-blocking) ===
#
# `uvicorn.run(...)` is a blocking call that takes over the asyncio
# event loop. In Jupyter / Colab / Kaggle the kernel already owns an
# event loop, so the naive form either deadlocks or silently exits.
# nest_asyncio.apply() patches the loop but doesn't fix the blocking
# problem — the cell stays "Running" forever and the user can't
# verify the server actually came up.
#
# This pattern starts uvicorn in a daemon background thread with its
# own event loop, then probes /health from the kernel thread to
# confirm it bound to the port. The cell completes in ~3 seconds and
# the server keeps serving for as long as the notebook session is
# alive. To stop it, interrupt this cell or restart the kernel.

import asyncio
import threading
import time

import requests
import uvicorn

config = uvicorn.Config(
    app,
    host='0.0.0.0',
    port=8000,
    log_level='info',
    loop='asyncio',
)
server = uvicorn.Server(config)

def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

_thread = threading.Thread(target=_run_server, name='uvicorn-bridge', daemon=True)
_thread.start()

# Poll /health locally until the server starts answering, or give up
# after ~10 seconds. This catches port conflicts, missing GPU drivers,
# and ONNX session-init crashes that would otherwise leave the user
# staring at a happy log and a dead tunnel.
deadline = time.time() + 10
healthy = None
while time.time() < deadline:
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=1)
        if r.ok:
            healthy = r.json()
            break
    except Exception:
        pass
    time.sleep(0.25)

if healthy:
    print(f"Bridge UP — model={healthy.get('model')} provider={healthy.get('provider')}")
    print('Server runs in the background; this cell is free. Use the tunnel cell above for the public URL.')
else:
    print('Bridge did NOT come up within 10s — check the cell output for an exception.')
    print('Common causes: another cell already on port 8000, model weights failed to load,')
    print('               CUDA / cuDNN mismatch (re-run cell 3 to confirm GPU init).')
